# Earth-like

SPEEDY T31L8 with realistic orography and a climatological surface: a relaxed slab ocean, SPEEDY's slab land, and a slab sea-ice model. This run is one command:

```bash
python -m jem.main +configuration=earth-slab
```

The terrain, the climatological forcing, the 30-day ocean relaxation, the 1-day land relaxation (jax-esm#1) and the sea ice seeded from the observed concentration are all set -- with the reason for each -- in `earth-slab.yaml`. This notebook builds the same configuration through `jem.configurations.load` -- the recipe door onto a validated `jem/config/configuration/*.yaml` (issue #131) -- rather than copying those tuned values into Python: hand-copying them here is exactly the drift the door exists to prevent, so the notebook reads them back out of `exp.config` instead.

## Build it

In [ ]:
from pathlib import Path

from jem import configurations, plot, run_chunked

output_dir = (Path("output") / "02-01_earth").resolve()
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
exp = configurations.load("earth-slab")
exp.coupler

The three tuned parameters `earth-slab.yaml` documents, read back from `exp.config` rather than restated here:

In [ ]:
{
    "ocean.params.relaxation_time [days]":
        exp.config["ocean"]["params"]["relaxation_time"] / 86400,
    "land.params.tdland [days]":
        exp.config["land"]["params"]["tdland"] / 86400,
    "seaice.ice_clim_file": exp.config["seaice"]["ice_clim_file"],
}

## Run it

In [ ]:
result = run_chunked(
    exp.coupler,
    **{
        **exp.run_kwargs,
        "output_dir": str(output_dir),
        "subsample": 3,       # 10 records out of 30 coupled days
        "checkpoint_path": None,
    },
)
result.steps_completed, [p.name for p in result.paths]

## What it wrote

One file per component per chunk, named after the coupled step its chunk starts on (`<component>-<first step>.nc`).

In [ ]:
atm_ds = plot.open_output(output_dir, "atm")
ocn_ds = plot.open_output(output_dir, "ocn")
lnd_ds = plot.open_output(output_dir, "lnd")
list(lnd_ds.data_vars)

## Plot

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

fig, axes = plt.subplots(
    1, 2, figsize=(13, 5), subplot_kw={"projection": ccrs.PlateCarree()}
)

sst = ocn_ds["sea_surface_temperature"].isel(time=-1) - 273.15
plot.map_plot(sst, ax=axes[0], coastlines=True,
              title="Sea surface temperature [°C]")

land_temperature = lnd_ds["land_surface_temperature"].isel(time=-1) - 273.15
plot.map_plot(land_temperature, ax=axes[1], coastlines=True,
              title="Land surface temperature [°C]")
plt.tight_layout()